<a href="https://colab.research.google.com/github/vipinkumarcp/llms_from_scratch/blob/main/peft_imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install  -q transformers peft accelerate datasets

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
print("training samples",len(dataset['train']))

training samples 25000


In [6]:
print("testing samples",len(dataset['test']))

testing samples 25000


In [7]:
#tokenizer

In [8]:
from transformers import AutoTokenizer


In [9]:
tokenizer  = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def tokenize(batch):
    result = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    result["labels"] = batch["label"]  # Keep the label for training
    return result


In [11]:
# Apply tokenization across entire dataset
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 50000
    })
})

In [13]:
# Set format for PyTorch
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [14]:
# Baseline Model Without LoRA
# Establish a baseline by loading DistilBERT without LoRA. This provides a reference point for comparing trainable parameters before and after adding LoRA adapters.

In [15]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Count trainable parameters in baseline
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable} / {total} ({100*trainable/total:.2f}%)")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable: 66955010 / 66955010 (100.00%)


In [16]:
# Apply LoRA Adapters
# Instead of updating every weight, LoRA injects small trainable matrices into attention layers. The base model stays frozen, and only these lightweight adapters get updated.

In [17]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                               # Rank: smaller = fewer params
    lora_alpha=16,                     # Scaling factor for LoRA updates
    target_modules=["q_lin", "v_lin"], # DistilBERT attention layers
    lora_dropout=0.1,                  # Regularization
    bias="none",                       # Don't add bias parameters
    task_type="SEQ_CLS"                # Sequence classification task
)

In [18]:
# Attach LoRA to Model
# Attach LoRA adapters to the model
lora_model = get_peft_model(model, lora_config)

# Show the dramatic reduction in trainable parameters
lora_model.print_trainable_parameters()

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [19]:
# Training with LoRA
# With LoRA attached, fine-tuning proceeds using the Hugging Face Trainer. Training on a subset demonstrates that LoRA can adapt efficiently even with limited data.

In [20]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
   eval_strategy="epoch" ,      # Check performance after each epoch
    num_train_epochs=1,                # Just 1 epoch for demo
    per_device_train_batch_size=8,     # 8 examples per batch
    per_device_eval_batch_size=8,
    logging_steps=50,                  # Log every 50 steps
    save_steps=2000,                   # Save checkpoint every 2000 steps
    learning_rate=5e-4,                # Higher than typical (faster demo)
    weight_decay=0.01,                 # L2 regularization
)

# Create small train/eval subsets
train_subset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_subset = dataset["test"].shuffle(seed=42).select(range(500))

# Initialize the Trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
)

# Run the training
print("Starting training with LoRA adapters...")
trainer.train()

Starting training with LoRA adapters...


Epoch,Training Loss,Validation Loss
1,0.280140,0.291401


TrainOutput(global_step=250, training_loss=0.3836898078918457, metrics={'train_runtime': 79.6574, 'train_samples_per_second': 25.108, 'train_steps_per_second': 3.138, 'total_flos': 269478813696000.0, 'train_loss': 0.3836898078918457, 'epoch': 1.0})

In [22]:
#
# Evaluate Outputs
# Compare predictions from the baseline model (before fine-tuning) and the LoRA fine-tuned model to see how training changed model behavior.

# Compare Model Predictions


In [23]:
import torch

text = "This movie was an absolute masterpiece. Brilliant performances!"

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt")

# Move inputs to same device as models
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Get predictions from both models
with torch.no_grad():  # Don't compute gradients during inference
    baseline_logits = model(**inputs).logits
    lora_logits = lora_model(**inputs).logits

# Convert logits to probabilities
baseline_probs = baseline_logits.softmax(-1)[0].cpu().numpy()
lora_probs = lora_logits.softmax(-1)[0].cpu().numpy()

print(f"Input: {text}")
print(f"\nBaseline predictions:")
print(f"  Negative: {baseline_probs[0]:.1%}")
print(f"  Positive: {baseline_probs[1]:.1%}")
print(f"\nLoRA fine-tuned predictions:")
print(f"  Negative: {lora_probs[0]:.1%}")
print(f"  Positive: {lora_probs[1]:.1%}")

Input: This movie was an absolute masterpiece. Brilliant performances!

Baseline predictions:
  Negative: 0.2%
  Positive: 99.8%

LoRA fine-tuned predictions:
  Negative: 0.2%
  Positive: 99.8%
